# Biohub Cell Tracking During Development

Detects nuclei in each 3D timepoint with a scale-normalized Laplacian-of-Gaussian response, then links centroids across consecutive timepoints in physical micrometre coordinates using a constrained assignment that permits 1-to-2 parent/child matches for divisions. Outputs `submission.csv` in the required node/edge schema.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, gaussian_laplace
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max


def _make_decompressor():
    try:
        from numcodecs import Blosc
        codec = Blosc()
        return lambda buf: codec.decode(buf)
    except Exception:
        pass
    try:
        import blosc as _b
        return lambda buf: _b.decompress(bytes(buf))
    except Exception:
        pass
    try:
        import blosc2 as _b2
        return lambda buf: _b2.decompress(bytes(buf))
    except Exception:
        pass
    try:
        import imagecodecs as _ic
        return lambda buf: _ic.blosc_decode(bytes(buf))
    except Exception:
        pass
    raise RuntimeError("no blosc decompressor available")


_decompress = _make_decompressor()

MAX_LINK_UM = 7.0
NUCLEUS_RADII_UM = (2.5, 3.5)
MIN_PEAK_DIST_UM = 1.2
MAD_THRESHOLD_K = 0.15
DEFAULT_SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)
MAX_DETECTIONS_PER_T = 6000
DIVISION_TOLERANCE_UM = 5.0
TEST_ROOT = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development/test")
OUT_PATH = Path("submission.csv")


def discover_datasets(test_root):
    if not test_root.exists():
        return []
    return sorted(p.name[:-5] for p in test_root.iterdir() if p.name.endswith(".zarr"))


def read_zarr_meta(zarr_root):
    level = zarr_root / "0"
    arr_meta = json.loads((level / "zarr.json").read_text())
    shape = tuple(arr_meta["shape"])
    chunks = tuple(arr_meta["chunk_grid"]["configuration"]["chunk_shape"])
    dtype_str = arr_meta["data_type"]
    dtype_map = {"uint16": "<u2", "uint8": "|u1", "uint32": "<u4", "int16": "<i2", "float32": "<f4"}
    dtype = np.dtype(dtype_map.get(dtype_str, "<u2"))
    scale = DEFAULT_SCALE.copy()
    group_meta_path = zarr_root / "zarr.json"
    if group_meta_path.exists():
        try:
            group_meta = json.loads(group_meta_path.read_text())
            ms = group_meta.get("attributes", {}).get("multiscales", [])
            if ms:
                ds_list = ms[0].get("datasets", [])
                if ds_list:
                    transforms = ds_list[0].get("coordinateTransformations", [])
                    for tr in transforms:
                        if tr.get("type") == "scale":
                            s = tr["scale"]
                            scale = np.array(s[-3:], dtype=np.float64)
                            break
        except Exception:
            pass
    return shape, chunks, dtype, scale


def _read_chunk(cp, dtype, chunk_spatial):
    if not cp.exists():
        return None
    raw = cp.read_bytes()
    decoded = _decompress(raw)
    return np.frombuffer(decoded, dtype=dtype).reshape(chunk_spatial)


def load_timepoint(zarr_root, t, shape, chunks, dtype):
    spatial_shape = shape[1:]
    chunk_spatial = chunks[1:]
    chunk_dims = [(s + c - 1) // c for s, c in zip(spatial_shape, chunk_spatial)]
    out = np.zeros(spatial_shape, dtype=dtype)
    tasks = []
    for zi in range(chunk_dims[0]):
        for yi in range(chunk_dims[1]):
            for xi in range(chunk_dims[2]):
                cp = zarr_root / "0" / "c" / str(t) / str(zi) / str(yi) / str(xi)
                tasks.append((zi, yi, xi, cp))
    with ThreadPoolExecutor(max_workers=4) as ex:
        results = list(ex.map(lambda args: (args[0], args[1], args[2], _read_chunk(args[3], dtype, chunk_spatial)), tasks))
    for zi, yi, xi, chunk_arr in results:
        if chunk_arr is None:
            continue
        z0 = zi * chunk_spatial[0]
        y0 = yi * chunk_spatial[1]
        x0 = xi * chunk_spatial[2]
        z1 = min(z0 + chunk_spatial[0], spatial_shape[0])
        y1 = min(y0 + chunk_spatial[1], spatial_shape[1])
        x1 = min(x0 + chunk_spatial[2], spatial_shape[2])
        out[z0:z1, y0:y1, x0:x1] = chunk_arr[: z1 - z0, : y1 - y0, : x1 - x0]
    return out

In [ ]:
def detect(vol, scale):
    v = vol.astype(np.float32)
    max_r = max(NUCLEUS_RADII_UM)
    smooth_sigma_z = max(max_r * 2.5 / scale[0], 1.0)
    smooth_sigma_xy = max(max_r * 2.5 / scale[1], 1.5)
    background = gaussian_filter(v, sigma=(smooth_sigma_z, smooth_sigma_xy, smooth_sigma_xy))
    v = np.maximum(v - background, 0)
    norm = float(np.percentile(v, 99.9))
    if norm <= 0:
        return np.zeros((0, 3), dtype=np.int64)
    v = v / norm

    response = None
    for r in NUCLEUS_RADII_UM:
        sz = max(r / scale[0] / np.sqrt(3.0), 0.6)
        sxy = max(r / scale[1] / np.sqrt(3.0), 0.8)
        resp = -gaussian_laplace(v, sigma=(sz, sxy, sxy)) * (sxy * sxy)
        response = resp if response is None else np.maximum(response, resp)

    pos = response[response > 0]
    if pos.size == 0:
        return np.zeros((0, 3), dtype=np.int64)
    med = float(np.median(pos))
    mad = 1.4826 * float(np.median(np.abs(pos - med))) + 1e-6
    threshold = med + MAD_THRESHOLD_K * mad

    min_dist = max(int(round(MIN_PEAK_DIST_UM / scale[1])), 1)
    coords = peak_local_max(
        response,
        min_distance=min_dist,
        threshold_abs=threshold,
        exclude_border=False,
    )
    if coords.size == 0:
        return coords.reshape(0, 3)
    scores = response[coords[:, 0], coords[:, 1], coords[:, 2]]
    order = np.argsort(-scores)
    coords = coords[order]
    if coords.shape[0] > MAX_DETECTIONS_PER_T:
        coords = coords[:MAX_DETECTIONS_PER_T]
    return coords

In [ ]:
def link_pair(prev_um, curr_um, predicted_um):
    if prev_um.shape[0] == 0 or curr_um.shape[0] == 0:
        return []
    d = np.linalg.norm(predicted_um[:, None, :] - curr_um[None, :, :], axis=-1)
    feasible = d <= MAX_LINK_UM
    if not feasible.any():
        return []
    big = MAX_LINK_UM * 100.0
    base = np.where(feasible, d, big)
    cost = np.vstack([base, base])
    n = prev_um.shape[0]
    rows, cols = linear_sum_assignment(cost)
    seen = set()
    parent_children = {}
    for r, c in zip(rows, cols):
        c_int = int(c)
        if cost[r, c] >= big or c_int in seen:
            continue
        parent = int(r % n)
        parent_children.setdefault(parent, []).append((float(cost[r, c]), c_int))
        seen.add(c_int)
    edges = []
    for parent, kids in parent_children.items():
        kids.sort()
        if len(kids) == 1:
            edges.append((parent, kids[0][1]))
        else:
            d0, c0 = kids[0]
            d1, c1 = kids[1]
            child_gap = float(np.linalg.norm(curr_um[c0] - curr_um[c1]))
            if d1 <= 0.75 * MAX_LINK_UM and child_gap <= DIVISION_TOLERANCE_UM * 2.0 and d1 - d0 <= DIVISION_TOLERANCE_UM:
                edges.append((parent, c0))
                edges.append((parent, c1))
            else:
                edges.append((parent, c0))
    return edges

In [ ]:
import os

WORKER_COUNT = max(1, min(4, (os.cpu_count() or 2)))


def _process_timepoint(args):
    zarr_root, t, shape, chunks, dtype, scale = args
    try:
        volume = load_timepoint(zarr_root, t, shape, chunks, dtype)
    except Exception:
        return t, np.zeros((0, 3), dtype=np.int64)
    coords = detect(volume, scale)
    return t, coords


def track_dataset(zarr_root):
    shape, chunks, dtype, scale = read_zarr_meta(zarr_root)
    T = shape[0]
    tasks = [(zarr_root, t, shape, chunks, dtype, scale) for t in range(T)]
    coords_by_t = {}
    with ThreadPoolExecutor(max_workers=WORKER_COUNT) as ex:
        for t, coords in ex.map(_process_timepoint, tasks):
            coords_by_t[t] = coords

    nodes = []
    edges = []
    next_id = 1
    prev_ids = []
    prev_um = np.zeros((0, 3), dtype=np.float64)
    prev_velocity_by_id = {}
    for t in range(T):
        coords = coords_by_t.get(t, np.zeros((0, 3), dtype=np.int64))
        ids = list(range(next_id, next_id + coords.shape[0]))
        next_id += coords.shape[0]
        for (z, y, x), nid in zip(coords, ids):
            nodes.append((nid, t, int(z), int(y), int(x)))
        curr_um = coords.astype(np.float64) * scale
        if t > 0 and prev_um.shape[0] and curr_um.shape[0]:
            if prev_velocity_by_id:
                velocity = np.array(
                    [prev_velocity_by_id.get(pid, [0.0, 0.0, 0.0]) for pid in prev_ids],
                    dtype=np.float64,
                )
            else:
                velocity = np.zeros_like(prev_um)
            predicted_um = prev_um + velocity
            links = link_pair(prev_um, curr_um, predicted_um)
            new_velocity_by_id = {}
            for i, j in links:
                edges.append((prev_ids[i], ids[j]))
                new_velocity_by_id[ids[j]] = (curr_um[j] - prev_um[i]).tolist()
            prev_velocity_by_id = new_velocity_by_id
        else:
            prev_velocity_by_id = {}
        prev_ids = ids
        prev_um = curr_um
    return nodes, edges

In [ ]:
rows = []
gid = 0
for ds in discover_datasets(TEST_ROOT):
    try:
        nodes, edges = track_dataset(TEST_ROOT / f"{ds}.zarr")
    except Exception:
        nodes, edges = [(1, 0, 0, 0, 0)], []
    if not nodes:
        nodes = [(1, 0, 0, 0, 0)]
    for nid, t, z, y, x in nodes:
        rows.append((gid, ds, "node", nid, t, z, y, x, -1, -1))
        gid += 1
    for s, d in edges:
        rows.append((gid, ds, "edge", -1, -1, -1, -1, -1, s, d))
        gid += 1

submission = pd.DataFrame(
    rows,
    columns=["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"],
)
submission.to_csv(OUT_PATH, index=False)
submission.head()